In [1]:

import os
import re
import numpy as np
import pandas as pd
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected!.")


PyTorch version: 2.12.0+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 3070 Ti Laptop GPU
GPU Memory: 8.2 GB


In [ ]:
MODEL_DIR = "models/whisper-finetuned/best_model/"
for dirname, _, filenames in os.walk(MODEL_DIR):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import transformers
import librosa
import whisper
import jiwer
from jiwer import wer
import gc

print(f"Transformers version: {transformers.__version__}")
print(f"Librosa version: {librosa.__version__}")
print(f"Whisper version: {whisper.__version__}")
print("Jiwer: OK")
print("All libraries installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 90.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 35.3 MB/s eta 0:00:00
Transformers version: 5.0.0
Librosa version: 0.11.0
Whisper version: 20250625
Jiwer: OK
All libraries installed successfully!


In [4]:
gc.collect()
torch.cuda.empty_cache()
print(f"GPU Free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0))/1e9:.2f} GB")

GPU Free: 15.64 GB


In [ ]:
def detect_language(text):
    if not isinstance(text, str):
        return 'unknown'
    # Tamil script
    if re.search(r'[\u0B80-\u0BFF]', text):
        return 'ta'
    # Hindi/Devanagari script
    if re.search(r'[\u0900-\u097F]', text):
        return 'hi'
    # Default to English/Latin
    return 'eng'

In [ ]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration, AutoModel
from tqdm import tqdm
import os
from dotenv import load_dotenv

load_dotenv()
HF_TOKEN = os.getenv('HF_TOKEN')
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load the fine-tuned model
print("Loading fine-tuned model...")
processor = WhisperProcessor.from_pretrained(MODEL_DIR)
model_inf = WhisperForConditionalGeneration.from_pretrained(MODEL_DIR)
model_inf = model_inf.to(device)
model_inf.eval()
print(f"Model loaded! GPU: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")

model = AutoModel.from_pretrained(
    "ai4bharat/indic-conformer-600m-multilingual",
    trust_remote_code=True,
    token=HF_TOKEN
)
model.eval()
model = model.to("cuda" if torch.cuda.is_available() else "cpu")
print(f"Model loaded! GPU: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")

Loading fine-tuned model...


Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

Model loaded! GPU: 1.62 GB


config.json:   0%|          | 0.00/241 [00:00<?, ?B/s]

model_onnx.py: 0.00B [00:00, ?B/s]

...

Please check FRAME_DURATION_MS. The timestamps can be inaccurate
Please check FRAME_DURATION_MS. The timestamps can be inaccurate


Fetching 404 files:   0%|          | 0/404 [00:00<?, ?it/s]

Please check FRAME_DURATION_MS. The timestamps can be inaccurate


/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:148: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Model loaded! GPU: 1.62 GB


In [ ]:
BASE_PATH = "audio_data"
TEST_AUDIO_PATH = f"{BASE_PATH}/test"
test_df = pd.read_csv(f"{BASE_PATH}/test.csv")
test_df['audio_path'] = test_df['audio'].apply(lambda x: os.path.join(TEST_AUDIO_PATH, x))

print(f"\nGenerating predictions for {len(test_df)} test files...")
print("="*50)

predictions = []
dtype = next(model_inf.parameters()).dtype
with torch.no_grad():
    for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
        try:
            # Load audio
            audio, _ = librosa.load(row['audio_path'], sr=16000)

            # Process
            input_features = processor(
                audio,
                sampling_rate=16000,
                return_tensors="pt"
            ).to(dtype).input_features.to(device)

            # Generate transcription
            predicted_ids = model_inf.generate(
                input_features,
                max_new_tokens=444,
                num_beams=5
            )

            # Decode
            transcription = processor.batch_decode(
                predicted_ids,
                skip_special_tokens=True
            )[0].strip()

            lang = detect_language(transcription)
            if lang != "eng":
                wav = torch.tensor(audio).unsqueeze(0)
                transcription = model(wav, lang, "rnnt").strip()
            predictions.append(transcription)

        except Exception as e:
            print(f"Error on {row['audio']}: {e}")
            predictions.append("")

# Create submission
print("\nCreating submission file...")
prediction_df = pd.DataFrame({
    'audio': test_df['audio'],
    'text': predictions
})

prediction_df.to_csv("audio_data/prediction.csv", index=False)
print("Submission saved to audio_data/prediction.csv")
print(f"\nSample predictions:")
for i in range(5):
    print(f"[{i}] {prediction_df['audio'].iloc[i]} → {prediction_df['text'].iloc[i][:80]}")


Generating predictions for 100 test files...



  0%|          | 0/100 [00:00<?, ?it/s]The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=444) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custo


Creating submission file...
Submission saved to /kaggle/working/submission.csv

Sample predictions:
[0] audio_00000.wav → इनमें जेपी मॉर्गन सिमस और ब्लैक रॉक के सीईओ शामिल हैं
[1] audio_00001.wav → एक हज़ार नौ सौ नब्बे में जहां दो मुस्लिम उम्मीदवारों ने जीत
[2] audio_00002.wav → மாட்டை வந்து மாட்டுப்பொங்கல் முடிஞ்சு மாட்டுப்பொங்கல் நைட்டு வந்து மாடை வந்து மா
[3] audio_00003.wav → அறிவிக்கிற மாதிரி ஒரு படை இருக்கணும்
[4] audio_00004.wav → and by this i knew the violence of my love i left the table without a moment's h
